# Heatwave-AI — Data & Model Pipeline (Colab)

Runs the **real** pipeline: Open-Meteo (ERA5) → sWBGT → per-province percentile + persistence labels → temporal split → LightGBM (isotonic-calibrated) → writes forecasts + thresholds to **Supabase** and uploads the model to **Hugging Face**.

### Set these Colab secrets first (🔑 left sidebar → Secrets, toggle “Notebook access”)
| Secret | Required? | What |
|---|---|---|
| `DATABASE_URL` | ✅ | Supabase **Session pooler** string (port 5432) — Supabase → Connect → Session pooler |
| `HF_TOKEN` | ✅ | Hugging Face **write** token |
| `GITHUB_TOKEN` | only if repo is private | GitHub PAT with `repo` scope |
| `OPENMETEO_API_KEY` | optional | paid Open-Meteo key → unlocks the full 1991-2025 range (free tier uses 2010-2025) |
| `OPENMETEO_START_YEAR` | optional | set to `1991` (needs the API key) for the full WMO baseline |
| `HF_REPO_ID` | optional | override the Hugging Face model repo |

## 1. Clone the repo (branch `feat/region-line-oa`)

In [ ]:
REPO = 'https://github.com/MCTEEKUNG/Heatwave_Backend_Elysia.git'
BRANCH = 'feat/region-line-oa'

tok = None
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

url = REPO.replace('https://', f'https://{tok}@') if tok else REPO
!rm -rf heatwave
!git clone --depth 1 -b $BRANCH $url heatwave
%cd heatwave

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt pyarrow 'psycopg[binary]' huggingface_hub

## 3. Load secrets into the environment

In [ ]:
import os
from google.colab import userdata

os.environ['DATABASE_URL'] = userdata.get('DATABASE_URL')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Optional secrets (ignored if not set)
for opt in ('OPENMETEO_API_KEY', 'OPENMETEO_START_YEAR', 'HF_REPO_ID'):
    try:
        v = userdata.get(opt)
        if v:
            os.environ[opt] = v
    except Exception:
        pass

print('DATABASE_URL set:', bool(os.environ.get('DATABASE_URL')))
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))
print('OPENMETEO_API_KEY set:', bool(os.environ.get('OPENMETEO_API_KEY')))

## 4. Build the dataset (real Open-Meteo)
Throttled + auto-retries on rate limits. **Default range 2010–2025** to fit the Open-Meteo free tier (the climatology auto-uses the fetched years). For the full 1991–2025 WMO baseline, set the `OPENMETEO_API_KEY` + `OPENMETEO_START_YEAR=1991` secrets. Takes a few minutes.

In [ ]:
!python -m pipeline.build_dataset

## 5. Train + calibrate + evaluate  —  🚦 QUALITY GATE
Read the printed metrics: **`test` PR-AUC / MCC / F2** must beat **`baseline_constant`**. If not, the model has no skill — add features (geopotential 500 hPa, soil moisture, more lags) before deploying.

In [ ]:
!python -m pipeline.train

## 6. Load thresholds into Supabase (`heatwave.province_thresholds`)

In [ ]:
!python -m pipeline.load_thresholds

## 7. Generate forecasts → Supabase (`heatwave.forecasts`)

In [ ]:
!python -m pipeline.run_forecast

## 8. Upload model + thresholds to Hugging Face (for Render to download in M4)

In [ ]:
import os
from huggingface_hub import HfApi

REPO_ID = os.environ.get('HF_REPO_ID', 'MCTEEKUNG123/Heatwave-AI')
api = HfApi(token=os.environ['HF_TOKEN'])

# Create the model repo if it doesn't exist yet (no-op if it already does).
api.create_repo(REPO_ID, repo_type='model', exist_ok=True)

api.upload_file(path_or_fileobj='models/heatwave_model.pkl',
                path_in_repo='models/heatwave_model.pkl', repo_id=REPO_ID)
api.upload_file(path_or_fileobj='data/processed/province_thresholds.parquet',
                path_in_repo='data/province_thresholds.parquet', repo_id=REPO_ID)
print('uploaded model + thresholds to', REPO_ID)

## Done ✅
- `heatwave.province_thresholds` + `heatwave.forecasts` populated in Supabase
- `heatwave_model.pkl` + thresholds parquet on Hugging Face

**Next:** verify the train metrics beat baseline (M2 gate), then M3 (LINE) → M4 (deploy).